# Polygon plot

In [ ]:
### import necessary libraries
import pandas as pd
import geopandas as gpd
import geojson

In [5]:
name_dir = 'SD1'

# dir = "D:\\Xenium"
dir = "E:\\Xenium_SD/"

dir_notebook = 'D:/Jupyter_notebook/Xenium_jupyter_notebook'

from module.misc import sample_name_import
samples_ids = sample_name_import(name_dir)

## Geojson preprocessing

### Import cells bondaries and create Geojson file (not resegmented)

In [6]:
for sample_id in samples_ids:
    ### Get cell and nucleus boundaries
    cells = pd.read_parquet(f"{dir}/{sample_id}/cell_boundaries.parquet")
    
    cells.groupby("cell_id")[['vertex_x', 'vertex_y']].agg({"vertex_x": list, "vertex_y": list}).reset_index().rename(columns={"vertex_x": "xs", "vertex_y": "ys"})
    cells['cell_id'] = sample_id + '_' + cells['cell_id']
    
    # Group the dataframe by the "Selection" column
    grouped = cells.groupby('cell_id')
    features = []
    
    for cell_id, group in grouped:
        coordinates = [(x, y) for x, y in zip(group['vertex_x'], group['vertex_y'])]
        if coordinates[0] != coordinates[-1]:
            coordinates.append(coordinates[0])
        
        polygon = geojson.Polygon([coordinates])
        feature = geojson.Feature(geometry=polygon, properties={"cell": cell_id})
        features.append(feature)
    
    feature_collection = geojson.FeatureCollection(features)
    
    with open(f'{dir_notebook}/coordinates/polygons/{sample_id}_cells.geojson', 'w') as f:
        geojson.dump(feature_collection, f)
    
    print(f"{sample_id}: GeoJSON saved")

SD1-ZT01: GeoJSON saved
SD1-ZT05: GeoJSON saved
SD1-ZT09: GeoJSON saved
SD1-ZT13: GeoJSON saved
SD1-ZT17: GeoJSON saved
SD1-ZT21: GeoJSON saved


### Update Geojson file (resegmented)

In [ ]:
### sample
sample = "3161-3__20240530__205547"
sample_ids = ['3160-1','3160-2','3159-1','2505-1','2505-2','2670-1']

In [100]:
for sample_id in sample_ids:
    cells_geo = gpd.read_file(f'/mnt/d/Xenium/H&E_staining/2024-04/Qupath_resegmentation/{sample_id}.geojson') ### if resegment
    cells_geo['cell'] = sample_id + '_' + cells_geo['id'] ### Rsegment only
    with open(f'coordinates/polygons/{sample_id}_cells.geojson', 'w') as f:
        geojson.dump(cells_geo, f)